<a href="https://colab.research.google.com/github/Gbriel26/IA_TEMA_ML/blob/main/IA_TEMA_ML_3_(Regresi%C3%B3n_Log%C3%ADstica).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

In [2]:
# Cargar el dataset
data = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Inspeccionar las primeras filas
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Identificar el tipo de datos en cada columna
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# Resumen estadístico de columnas numéricas
data.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
# Resumen estadístico de columnas object
data.describe(include='object').T

,count,unique,top,freq
customerID,7043,7043,3186-AJIEK,1
gender,7043,2,Male,3555
Partner,7043,2,No,3641
Dependents,7043,2,No,4933
PhoneService,7043,2,Yes,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095


In [6]:
# Inspeccionar valores únicos en columnas categóricas
categorical_cols = data.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"Valores únicos en {col}: {data[col].unique()}")

Valores únicos en customerID: ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
Valores únicos en gender: ['Female' 'Male']
Valores únicos en Partner: ['Yes' 'No']
Valores únicos en Dependents: ['No' 'Yes']
Valores únicos en PhoneService: ['No' 'Yes']
Valores únicos en MultipleLines: ['No phone service' 'No' 'Yes']
Valores únicos en InternetService: ['DSL' 'Fiber optic' 'No']
Valores únicos en OnlineSecurity: ['No' 'Yes' 'No internet service']
Valores únicos en OnlineBackup: ['Yes' 'No' 'No internet service']
Valores únicos en DeviceProtection: ['No' 'Yes' 'No internet service']
Valores únicos en TechSupport: ['No' 'Yes' 'No internet service']
Valores únicos en StreamingTV: ['No' 'Yes' 'No internet service']
Valores únicos en StreamingMovies: ['No' 'Yes' 'No internet service']
Valores únicos en Contract: ['Month-to-month' 'One year' 'Two year']
Valores únicos en PaperlessBilling: ['Yes' 'No']
Valores únicos en PaymentMethod: ['Electronic check' 'Maile

In [8]:
# Codificar variables categóricas
label_encoder = LabelEncoder()
for col in categorical_cols:
    data[col] = label_encoder.fit_transform(data[col])

In [9]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,5375,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,29.85,2505,0
1,3962,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,56.95,1466,0
2,2564,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,53.85,157,1
3,5535,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,42.30,1400,0
4,6511,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,70.70,925,1


In [10]:
scaler = StandardScaler()
data['MonthlyCharges'] = scaler.fit_transform(data[['MonthlyCharges']])
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,5375,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,-1.160323,2505,0
1,3962,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,-0.259629,1466,0
2,2564,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,-0.362660,157,1
3,5535,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,-0.746535,1400,0
4,6511,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,0.197365,925,1


In [11]:
X = data.drop(columns=['customerID', 'Churn'])
y = data['Churn']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [13]:
# Crear el modelo
model = LogisticRegression()

# Entrenar el modelo
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
# Exactitud
print("Exactitud:", accuracy_score(y_test, y_pred))

# Matriz de confusión
print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred))

# Informe de clasificación
print("Informe de Clasificación:\n", classification_report(y_test, y_pred))

Exactitud: 0.8078561287269286
Matriz de Confusión:
 [[1398  141]
 [ 265  309]]
Informe de Clasificación:
               precision    recall  f1-score   support

           0       0.84      0.91      0.87      1539
           1       0.69      0.54      0.60       574

    accuracy                           0.81      2113
   macro avg       0.76      0.72      0.74      2113
weighted avg       0.80      0.81      0.80      2113



In [15]:
# Datos simulados
new_data = pd.DataFrame({
    'gender': [1, 0],
    'SeniorCitizen': [0, 1],
    'Partner': [1, 0],
    'Dependents': [0, 0],
    'tenure': [12, 36],
    'PhoneService': [1, 1],
    'MultipleLines': [0, 1],
    'InternetService': [1, 2],
    'OnlineSecurity': [0, 1],
    'OnlineBackup': [1, 0],
    'DeviceProtection': [0, 1],
    'TechSupport': [1, 0],
    'StreamingTV': [1, 0],
    'StreamingMovies': [0, 1],
    'Contract': [0, 1],
    'PaperlessBilling': [1, 1],
    'PaymentMethod': [2, 3],
    'MonthlyCharges': scaler.transform([[70], [80]]).flatten(),
    'TotalCharges': [840, 2880]
})

# Predicciones
predictions = model.predict(new_data)
print("Predicciones para nuevos datos:", predictions)

Predicciones para nuevos datos: [0 0]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
